# 02 · Was the experiment run correctly?

Before looking at outcomes, two checks from the analysis plan:
1. **Sample ratio mismatch (SRM):** are group sizes consistent with a random equal split?
2. **Covariate balance:** were the groups alike *before* the email?

If either fails, differences in outcomes could come from how customers were assigned rather than from the email.

In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
import ab_test as ab

pd.set_option("display.float_format", "{:,.4f}".format)
ab.apply_chart_style()

In [3]:
engine = ab.get_engine()
df = ab.load_clean(engine)
df.shape

MySQL password for root:  ········


(64000, 13)

## 1. Sample ratio mismatch

In [4]:
srm = ab.srm_check(df)
print(srm["observed"].to_string())
print(f"\nChi-square = {srm['chi2']:.3f}, p-value = {srm['p_value']:.3f} -> {'PASS' if srm['passed'] else 'FAIL'}")

test_group
Control    21306
Mens       21307
Womens     21387

Chi-square = 0.203, p-value = 0.904 -> PASS


The Womens group has about 80 more customers. With a correct random split, a gap this size or larger happens by chance about 90% of the time (p = 0.90), so there is no sign of an assignment problem.

## 2. Covariate balance

In [5]:
balance = ab.covariate_balance(df)
print(f"Largest |SMD|: {balance['max_abs_smd'].max():.3f} (threshold {ab.SMD_THRESHOLD})")
balance.round(3)

Largest |SMD|: 0.014 (threshold 0.1)


,Mens,Womens,max_abs_smd
area_Rural,0.0140,0.0040,0.0140
area_Suburban,-0.0120,-0.0010,0.0120
channel_Web,0.0110,-0.0050,0.0110
past_purchase_Womens only,0.0050,0.0090,0.0090
channel_Phone,-0.0080,0.0090,0.0090
history,0.0080,0.0070,0.0080
past_purchase_Mens only,-0.0080,-0.0050,0.0080
recency,0.0070,0.0050,0.0070
past_purchase_Both,0.0050,-0.0060,0.0060
channel_Multichannel,-0.0040,-0.0050,0.0050


Every pre-campaign characteristic differs by less than 0.015 standard deviations between groups, well under the 0.1 threshold. The groups were alike before the email, so later differences can be attributed to the email.

In [6]:
max_smd = balance["max_abs_smd"].max()
result_validity = pd.DataFrame([
    {"check": "sample ratio mismatch (chi-square)", "statistic": srm["chi2"],
     "p_value": srm["p_value"], "result": "PASS" if srm["passed"] else "FAIL"},
    {"check": "max absolute standardised mean difference", "statistic": max_smd,
     "p_value": None, "result": "PASS" if max_smd <= ab.SMD_THRESHOLD else "FAIL"},
])
ab.save_tables({"result_validity": result_validity}, engine)

result_validity                2 rows saved
